# SANAD Network RC3 — LensFD FULL ARCHIVE REPAIR (v2)

This fixes the **`tarfile.ReadError: unexpected end of data`** seen near member 113 (`no-s2n.hdf5`, 21,589,436,536 bytes). It first diagnoses the previous `members.jsonl`, then validates the entire TAR/GZIP (including its tail), repairs a corrupt download using HTTP byte-range resume and full integrity testing, and audits **100%** of every HDF5 numeric dataset. Existing 112+ successful member checks are preserved and, if the archive changes, safely revalidated against SHA-256. **No sampling, no skipped files.**

The error can mean an incomplete archive and/or insufficient scratch space. The error of the large member itself is in `members.jsonl`; this notebook checks both. It does not modify RC3 model weights. It cannot invent the required RC3 7-array/300-period evidence from raw LensFD IQ data.

**License:** Keep Rice LensFD raw data private. Archive source: https://renew.rice.edu/dataset-lens.html.


In [ ]:
# Install only tools required to inspect the entire real measurement archive.
%pip -q install 'numpy>=1.26,<3' 'h5py>=3.10,<5' 'requests>=2.31,<3' 'scikit-learn>=1.5,<1.9' 'joblib>=1.3,<2' 'pytest>=8,<10'
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, time, requests
ROOT = Path('/content/drive/MyDrive/SANAD_Network/RC3_LensFD_FULL')
RAW = ROOT / 'source_archive_PRIVATE'
RESULTS = ROOT / 'results_full'
RAW.mkdir(parents=True,exist_ok=True)
RESULTS.mkdir(parents=True,exist_ok=True)
SCRATCH = Path('/content')
print('Drive destination:',ROOT)
print('Colab scratch free, GiB:',round(shutil.disk_usage(SCRATCH).free/(2**30),2))
print('Raw LensFD is licensed; keep this Drive folder private.')

In [ ]:
# Full-data scanner v2: patched tar handling, revalidation, and incomplete-run reporting.
from pathlib import Path
Path('/content/lensfd_full_run.py').write_text('"""Full-archive LensFD source audit and frozen RC3 compatibility gate.\n\nRead every byte of every regular archive member and every element in every numeric\nHDF5 dataset. No sampling, no silent truncation, no fabricated RC3 features.\nSource: Rice University LensFD (Chen et al., IEEE TVT 2023).\nLicense: Rice grants non-exclusive/non-transferable use; raw data must not be redistributed.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nimport tarfile\nimport tempfile\nimport subprocess\nfrom datetime import datetime, timezone\nfrom itertools import product\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Iterator\n\nimport numpy as np\n\nBLOCK_BYTES = 16 * 1024 * 1024\nOFFICIAL_URL = \'https://renew.rice.edu/datasets/ashu/traces/LensFD/data_lensFD.tgz\'\nEXPECTED_FROZEN_COMMIT = \'182e77b485cceb770ee6570ba332205c73823c91\'\nEXPECTED_MODEL_SHA256 = \'0a90835bb57a56437fc09211902b581bb22377676f91dce68f309c92e3a95bf4\'\n\n\ndef utcnow():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef hash_file(p: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:\n    h = hashlib.sha256()\n    with Path(p).open(\'rb\') as file:\n        for b in iter(lambda: file.read(chunk_bytes), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef atomic_json(path: Path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + \'.tmp\')\n    with temp.open(\'w\', encoding=\'utf8\') as f:\n        json.dump(obj, f, indent=2, ensure_ascii=False, allow_nan=False)\n        f.flush(); os.fsync(f.fileno())\n    os.replace(temp, path)\n\n\ndef _json_line(path: Path, obj):\n    with Path(path).open(\'a\', encoding=\'utf8\') as f:\n        f.write(json.dumps(obj, ensure_ascii=False, allow_nan=False) + \'\\n\')\n        f.flush(); os.fsync(f.fileno())\n\n\ndef iter_dataset_slices(shape, itemsize, target_bytes=BLOCK_BYTES) -> Iterator[tuple]:\n    """Cover the complete N-D logical HDF5 space exactly once, at bounded memory."""\n    if not shape:\n        yield ()\n        return\n    blocks = list(map(int, shape))\n    if any(k <= 0 for k in blocks):\n        return\n    while math.prod(blocks) * max(1, itemsize) > target_bytes:\n        largest = max(range(len(blocks)), key=lambda i: blocks[i])\n        if blocks[largest] == 1:\n            raise MemoryError(\'Single HDF5 element larger than target block\')\n        blocks[largest] = (blocks[largest] + 1) // 2\n    ranges = [range(0, n, w) for n,w in zip(shape, blocks)]\n    for start in product(*ranges):\n        yield tuple(slice(s, min(s+b,n)) for s,b,n in zip(start,blocks,shape))\n\n\ndef _scan_numeric(ds) -> dict:\n    dtype = ds.dtype\n    count = int(math.prod(ds.shape)) if ds.shape else 1\n    record = dict(shape=list(ds.shape), dtype=str(dtype), values=count)\n    if dtype.kind not in (\'b\',\'i\',\'u\',\'f\',\'c\'):\n        record.update(numeric_analyzed=False, values_scanned=0, reason=\'Non-plain-numeric HDF5 dtype; complete source bytes still SHA-256 verified\')\n        return record\n    total = 0\n    zero = 0\n    finite = 0\n    abs_sum = 0.0\n    sq_sum = 0.0\n    mn = float(\'inf\'); mx = float(\'-inf\')\n    for sl in iter_dataset_slices(ds.shape, int(dtype.itemsize)):\n        vals = np.asarray(ds[sl] if sl else ds[()]).reshape(-1)\n        total += len(vals)\n        valid = np.isfinite(vals)\n        finite += int(np.count_nonzero(valid))\n        good = vals[valid]\n        if good.size:\n            mag = np.abs(good.astype(np.complex128 if np.iscomplexobj(good) else np.float64))\n            zero += int(np.count_nonzero(mag == 0))\n            abs_sum += float(np.sum(mag, dtype=np.float64))\n            sq_sum += float(np.sum(mag * mag, dtype=np.float64))\n            mn = min(mn, float(np.min(mag)))\n            mx = max(mx, float(np.max(mag)))\n    if total != count:\n        raise RuntimeError(f\'Numeric coverage incomplete {ds.name}: {total}/{count}\')\n    record.update(numeric_analyzed=True,values_scanned=total,finite_values=finite,\n                  nonfinite_values=total-finite,zero_values=zero,\n                  mean_abs=abs_sum/finite if finite else None,\n                  rms_abs=math.sqrt(sq_sum/finite) if finite else None,\n                  min_abs=mn if finite else None,max_abs=mx if finite else None)\n    return record\n\n\ndef inspect_hdf5(path: Path):\n    import h5py\n    rows=[]; n_numeric=0; n_analyzed=0; total_values=0\n    with h5py.File(path, \'r\') as f:\n        def visit(name, item):\n            nonlocal n_numeric,n_analyzed,total_values\n            if not isinstance(item,h5py.Dataset):\n                return\n            stats=_scan_numeric(item)\n            stats[\'dataset_path\']=\'/\'+name\n            stats[\'attribute_names\']=sorted(list(item.attrs.keys()))\n            rows.append(stats)\n            if stats[\'numeric_analyzed\']:\n                n_numeric+=1\n                n_analyzed+=stats[\'values_scanned\']\n            total_values+=stats[\'values\']\n        f.visititems(visit)\n        root_attrs=sorted(list(f.attrs.keys()))\n    return {\'hdf5_datasets\':rows,\'hdf5_dataset_count\':len(rows),\n            \'numeric_dataset_count\':n_numeric,\'numeric_values_scanned\':n_analyzed,\n            \'total_logical_values\':total_values,\'root_attribute_names\':root_attrs}\n\n\ndef _safe_member_name(name):\n    p=Path(name)\n    if p.is_absolute() or \'..\' in p.parts or \'\\\\\' in name or not name or name.startswith(\'~\'):\n        raise ValueError(\'Unsafe archive member path\')\n    return str(p)\n\n\ndef _load_journal(journal):\n    if not journal.exists():return {}\n    records={}\n    with journal.open(encoding=\'utf8\') as f:\n        for line in f:\n            if line.strip():\n                row=json.loads(line)\n                # Append-only retry journal; most recent attempt for each member wins.\n                records[row[\'name\']]=row\n    return records\n\n\ndef validate_archive_structure(archive: str | Path) -> dict:\n    """Exhaustively validate gzip CRC and TAR member framing without extracting data.\n\n    GNU tar streams through all bytes, so this checks the large final member too.\n    Merely hashing an incomplete file does NOT prove completeness.\n    """\n    archive = Path(archive)\n    check = subprocess.run([\'tar\', \'-tzf\', str(archive)], stdout=subprocess.DEVNULL,\n                           stderr=subprocess.PIPE, text=True, errors=\'replace\')\n    result = {\'archive\': str(archive), \'archive_size\': archive.stat().st_size,\n              \'tar_exit_code\': check.returncode,\n              \'complete\': check.returncode == 0,\n              \'stderr_tail\': check.stderr[-2400:]}\n    return result\n\n\ndef archive_preflight(archive, output, *, require_complete=True):\n    """Prevent a slow HDF5 scan from ending on a truncated archive."""\n    output = Path(output); output.mkdir(parents=True,exist_ok=True)\n    report = validate_archive_structure(archive)\n    atomic_json(output/\'ARCHIVE_PREFLIGHT.json\', report)\n    if not report[\'complete\'] and require_complete:\n        raise RuntimeError(\'LensFD TAR/GZIP integrity FAILED before HDF5 scanning. \'\n                           \'Inspect ARCHIVE_PREFLIGHT.json; repair/re-download the \'\n                           \'source archive, then rerun. Existing members.jsonl \'\n                           \'and completed HDF5 statistics are preserved. \'\n                           + report[\'stderr_tail\'][-500:])\n    return report\n\n\ndef _archive_summary(*, archive, arch_sha, seen, prior, dirs, errors, source_changed):\n    regular = [r for r in prior.values() if r[\'name\'] in seen]\n    ok = [r for r in regular if r[\'status\'] == \'OK\']\n    complete = (len(regular) == len(seen) and len(ok) == len(seen) and not errors)\n    source_groups = Counter(Path(r[\'name\']).name.split(\'-\')[0].lower() for r in ok)\n    return {\'status\':\'COMPLETE_FULL_ARCHIVE_AUDIT\' if complete else \'INCOMPLETE\',\n       \'official_source\':OFFICIAL_URL, \'finished_utc\':utcnow(),\n       \'archive_sha256\':arch_sha, \'archive_bytes_hashed\':archive.stat().st_size,\n       \'archive_members_regular\':len(seen), \'archive_members_directories\':dirs,\n       \'source_filename_prefix_counts\':dict(sorted(source_groups.items())),\n       \'members_checked\':len(regular), \'members_ok\':len(ok),\n       \'member_bytes_accounted\':sum(r[\'size\'] for r in ok),\n       \'hdf5_dataset_count\':sum(r.get(\'hdf5_dataset_count\',0) for r in ok),\n       \'hdf5_numeric_values_scanned\':sum(r.get(\'numeric_values_scanned\',0) for r in ok),\n       \'hdf5_total_logical_values\':sum(r.get(\'total_logical_values\',0) for r in ok),\n       \'hdf5_nonnumeric_dataset_count\':sum(r.get(\'hdf5_dataset_count\',0)-r.get(\'numeric_dataset_count\',0) for r in ok),\n       \'bytes_sampled\':0, \'inference_source\':\'RAW_REAL_RF_MEASUREMENTS\',\n       \'rc3_qualification_on_lensfd\':\'NOT_YET_TESTED\',\n       \'archive_replacement_detected\':bool(source_changed),\n       \'errors\':errors + [{\'name\':r[\'name\'],\'error\':r.get(\'error\',\'unresolved\')}\n                          for r in regular if r[\'status\']!=\'OK\' and r[\'name\'] not in {e[\'name\'] for e in errors}]}\n\n\ndef run_full_archive(archive: str | Path, output: str | Path, scratch_dir: str | Path | None = None,\n                     strict=True, *, preflight=True) -> dict:\n    """Audit every regular TAR member; preserve checkpoints, including on TAR read errors.\n\n    Reused source: completed HDF5 member checks are not repeated.\n    Replaced source: re-hash cached members BEFORE reusing their HDF5 statistics.\n    No sample limits, no silent skipping.  The frozen RC3 model is not changed.\n    """\n    archive = Path(archive); output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    if not archive.is_file(): raise FileNotFoundError(str(archive))\n    if preflight: archive_preflight(archive, output)\n    arch_size=archive.stat().st_size\n    arch_sha=hash_file(archive)\n    lock=output/\'SOURCE_LOCK.json\'\n    previous=json.loads(lock.read_text()) if lock.exists() else None\n    source_changed=bool(previous and\n        (previous[\'archive_sha256\'],previous[\'archive_size\'])!=(arch_sha,arch_size))\n    # Do NOT discard old journal simply because source was repaired; verify each\n    # cached member\'s SHA against the replacement archive before trusting stats.\n    journal=output/\'members.jsonl\'; prior=_load_journal(journal)\n    seen=set();errors=[];dirs=0;tar_error=None\n    scratch=Path(scratch_dir or tempfile.gettempdir()); scratch.mkdir(parents=True,exist_ok=True)\n    try:\n        with tarfile.open(archive,\'r|gz\') as tar:\n            for member in tar:\n                name=_safe_member_name(member.name)\n                if member.isdir(): dirs+=1; continue\n                if name in seen: raise RuntimeError(f\'Duplicate archive member {name}\')\n                seen.add(name)\n                old=prior.get(name)\n                if old and old[\'status\']==\'OK\' and old[\'size\']==member.size and not source_changed:\n                    continue\n                row={\'name\':name, \'size\':member.size,\'mtime\':member.mtime,\'status\':\'ERROR\',\n                     \'archive_sha256\':arch_sha}\n                tempname=None\n                try:\n                    if not member.isfile():\n                        raise ValueError(f\'Unsupported non-regular archive object: {member.type!r}\')\n                    inp=tar.extractfile(member)\n                    if inp is None:raise RuntimeError(\'Could not access archive member stream\')\n                    # If source archive was repaired, verify old 112+ completed members\n                    # by hashing them directly without extracting / re-scanning HDF5.\n                    if old and old[\'status\']==\'OK\' and old[\'size\']==member.size and source_changed:\n                        dig=hashlib.sha256(); remaining=member.size\n                        while remaining:\n                            b=inp.read(min(8*1024*1024,remaining))\n                            if not b: raise EOFError(f\'Truncated archive member {name}\')\n                            dig.update(b); remaining-=len(b)\n                        if dig.hexdigest()!=old.get(\'sha256\'):\n                            raise RuntimeError(\'PREVIOUS_OK_FILE_CHANGED; cannot reuse its HDF5 statistics. \'\n                                               \'Use a fresh audit folder for this different source.\')\n                        row={**old,\'archive_sha256\':arch_sha,\n                             \'cache_verified_against_repaired_archive\':True}\n                    else:\n                        free=shutil.disk_usage(str(scratch)).free\n                        margin=256*1024*1024\n                        if member.size+margin>free:\n                            raise OSError(\'INSUFFICIENT_SCRATCH_DISK: \'\n                               f\'{name} is {member.size:,} bytes ({member.size/2**30:.2f} GiB); \'\n                               f\'/content has only {free:,} bytes ({free/2**30:.2f} GiB) free; \'\n                               \'select a larger-disk Colab runtime. NO SUBSAMPLING.\')\n                        dig=hashlib.sha256()\n                        with tempfile.NamedTemporaryFile(prefix=\'lensfd_\',\n                            suffix=Path(name).suffix,dir=str(scratch),delete=False) as temp:\n                            tempname=temp.name; remaining=member.size\n                            while remaining:\n                                b=inp.read(min(8*1024*1024,remaining))\n                                if not b: raise EOFError(f\'Truncated archive member {name}\')\n                                temp.write(b);dig.update(b);remaining-=len(b)\n                        row[\'sha256\']=dig.hexdigest()\n                        if Path(name).suffix.lower() in (\'.hdf5\',\'.h5\'):\n                            row.update(inspect_hdf5(Path(tempname)))\n                            row[\'analysis\']=\'FULL_HDF5_NUMERIC_SCAN\'\n                        else:\n                            row[\'analysis\']=\'FULL_BYTE_SHA256_ONLY\'\n                            row[\'warning\']=\'Non-HDF5 member fully SHA-256 hashed\'\n                        row[\'status\']=\'OK\'\n                except Exception as exc:\n                    row.update(status=\'ERROR\',error=f\'{type(exc).__name__}: {exc}\')\n                    errors.append({\'name\':name,\'error\':row[\'error\']})\n                finally:\n                    if tempname is not None:Path(tempname).unlink(missing_ok=True)\n                _json_line(journal,row); prior[name]=row\n                print(f\'[{len(seen)}] {row["status"]:5} {name}  bytes={member.size:,}\',flush=True)\n    except (tarfile.TarError, EOFError, OSError, RuntimeError, ValueError) as exc:\n        tar_error={\'name\':\'__ARCHIVE_STREAM__\',\'error\':f\'{type(exc).__name__}: {exc}\'}\n        errors.append(tar_error)\n    summary=_archive_summary(archive=archive, arch_sha=arch_sha, seen=seen,\n                             prior=prior,dirs=dirs,errors=errors,source_changed=source_changed)\n    if tar_error: summary[\'fatal_tar_stream_error\']=tar_error[\'error\']\n    atomic_json(output/\'FULL_AUDIT_SUMMARY.json\',summary)\n    if summary[\'status\']==\'COMPLETE_FULL_ARCHIVE_AUDIT\':\n        if source_changed and previous:\n            atomic_json(output/\'SOURCE_LOCK_PREVIOUS.json\',previous)\n        atomic_json(lock, {\'official_source\':OFFICIAL_URL,\'archive_size\':arch_size,\n                           \'archive_sha256\':arch_sha,\'verified_complete_utc\':utcnow()})\n    if strict and summary[\'status\']!=\'COMPLETE_FULL_ARCHIVE_AUDIT\':\n        detail=\'; \'.join(x[\'error\'] for x in summary[\'errors\'][-3:])\n        raise RuntimeError(\'Full LensFD archive audit INCOMPLETE. See \'\n                           \'FULL_AUDIT_SUMMARY.json and members.jsonl. \'+detail)\n    return summary\n\n\ndef rc3_contract_report(output_dir:str|Path, frozen_repository_path:str|Path|None=None):\n    """Fail closed: LensFD raw IQ is NOT the frozen RC3 300x7 observation trace.\n\n    RC3 requires real upstream evidence streams produced by the frozen Sionna\n    feature-generation pathway, not arbitrary tensor reshaping or synthetic rows.\n    """\n    reasons=[\n        \'RC3 input is an evidence NPZ of 300 reporting periods x 7 arrays with precomputed v11, fast/strict and temporal delta observations, not raw HDF5 IQ/CSI.\',\n        \'LensFD self-interference recording covers up to 32 antenna ports in any run; seven eight-element RC3 arrays must not be fabricated from asynchronous recordings.\',\n        \'LensFD source does not supply the frozen RC3 temporal feature stream, matching clock, 7-array baselines, or a machine-readable verified fault-label map by default.\',\n        \'Transforming raw IQ/CSI into plausible arrays or random RC3 observation features would not constitute an RC3 evaluation.\',\n    ]\n    out={\'dataset\':\'Rice LensFD\',\'mode\':\'FROZEN_RC3\',\'decision_threshold\':0.50,\n         \'expected_frozen_commit\':EXPECTED_FROZEN_COMMIT,\n         \'model_sha256\':EXPECTED_MODEL_SHA256,\'input_contract\':\'NPZ [2,300,7,3] plus [300,7,3] and v11 [300,7] per stream\',\n         \'adapter_status\':\'NO_VALIDATED_LENSFD_TO_RC3_FEATURE_ADAPTER\',\n         \'rc3_predictions_produced\':0,\'rc3_field_fault_detection_rate\':None,\n         \'reasons\':reasons,\'integrity_check_status\':\'SEPARATE_FROM_LENSFD_INFERENCE\'}\n    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)\n    atomic_json(output_dir/\'RC3_COMPATIBILITY.json\',out)\n    return out\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--archive\',required=True)\n    parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--scratch\',default=\'/content\')\n    args=parser.parse_args()\n    print(json.dumps(run_full_archive(args.archive,args.output,args.scratch),indent=2))\n    print(json.dumps(rc3_contract_report(args.output),indent=2))\n',encoding='utf-8')
import importlib, lensfd_full_run
importlib.reload(lensfd_full_run)
from lensfd_full_run import run_full_archive, archive_preflight, validate_archive_structure, rc3_contract_report
print('Loaded repaired full-data runner v2')


## 1. Review previous failure and disk capacity

In [ ]:
# Inspect old checkpoint records (the 112 completed files remain on Drive).
import json, shutil
journal=RESULTS/'members.jsonl'
if journal.exists():
    last=journal.read_text(encoding='utf-8').splitlines()[-5:]
    print('Most recent audit entries:')
    for line in last:
        row=json.loads(line)
        print(row['status'],row['name'],row.get('error',''))
else:
    print('No prior journal found; starting a fresh full audit.')
print('Scratch disk available:',f'{shutil.disk_usage(SCRATCH).free/2**30:.2f} GiB')
print('Largest known extracted member: 21,589,436,536 bytes = 20.11 GiB')
if shutil.disk_usage(SCRATCH).free < 21_589_436_536 + 256*2**20:
    print('DISK WARNING: This runtime cannot extract the largest member; select a larger-disk runtime before the full HDF5 scan.')


## 2. Repair and validate the entire publisher archive (no subsampling)

In [ ]:
# Full-archive verified downloader. No subsampling; retains all journal records.
from pathlib import Path
import requests, subprocess, time, os, shutil, json
from lensfd_full_run import validate_archive_structure, atomic_json
URL = 'https://renew.rice.edu/datasets/ashu/traces/LensFD/data_lensFD.tgz'
ARCHIVE = RAW/'data_lensFD.tgz'
PART = RAW/'data_lensFD.tgz.part'
AUTO_REPLACE_INVALID_PUBLIC_ARCHIVE = True  # Raw archive is reproducibly downloadable; journal remains intact.

def _remote_metadata(url):
    with requests.Session() as session:
        try:
            r=session.head(url,allow_redirects=True,headers={'Accept-Encoding':'identity'},timeout=(30,90))
            if not r.ok:return (None,None)
            total=int(r.headers['Content-Length']) if r.headers.get('Content-Length','').isdigit() else None
            return total,r.headers.get('ETag')
        except requests.RequestException:return (None,None)

def _stream_with_resume(url,part,expected,etag,max_attempts=8):
    part=Path(part)
    for attempt in range(1,max_attempts+1):
        offset=part.stat().st_size if part.exists() else 0
        if expected is not None and offset>expected:
            print('Oversized .part; restarting cleanly.');part.unlink();offset=0
        if expected is not None and offset==expected:
            print('Part file reached publisher byte count.');return
        headers={'Accept-Encoding':'identity'}
        if offset:
            headers['Range']=f'bytes={offset}-'
            if etag and not etag.startswith('W/'):headers['If-Range']=etag
        print(f'Download attempt {attempt}/{max_attempts}, offset {offset:,}',flush=True)
        try:
            with requests.get(url,headers=headers,stream=True,timeout=(60,180),allow_redirects=True) as r:
                r.raise_for_status()
                if offset and r.status_code==206:
                    cr=r.headers.get('Content-Range','')
                    if not cr.startswith(f'bytes {offset}-'):
                        raise RuntimeError('Incorrect HTTP Range response: '+cr)
                    total=cr.rsplit('/',1)[-1]
                    if total.isdigit():
                        total=int(total)
                        if expected is not None and expected!=total:
                            raise RuntimeError('Source total changed during resume')
                        expected=total
                    mode='ab'
                elif r.status_code==200:
                    if offset:print('Server ignored Range: safely restarting from zero.')
                    expected=int(r.headers['Content-Length']) if r.headers.get('Content-Length','').isdigit() else expected
                    mode='wb';offset=0
                elif r.status_code==206 and offset==0:
                    cr=r.headers.get('Content-Range','')
                    if not cr.startswith('bytes 0-'):raise RuntimeError('Wrong response for initial range')
                    total=cr.rsplit('/',1)[-1]
                    if total.isdigit():expected=int(total)
                    mode='wb'
                else:raise RuntimeError('Unexpected HTTP '+str(r.status_code))
                last=offset
                with part.open(mode) as out:
                    for b in r.iter_content(8*1024*1024):
                        if not b:continue
                        out.write(b);offset+=len(b)
                        if offset-last>=512*2**20:
                            out.flush();os.fsync(out.fileno());last=offset
                            print('Downloaded',f'{offset/2**30:.2f} GiB',flush=True)
                    out.flush();os.fsync(out.fileno())
            if expected is None or part.stat().st_size==expected:return
            print(f'Short transfer {part.stat().st_size:,}/{expected:,}; retrying Range')
        except (requests.RequestException,OSError) as exc:
            print('Download interrupted:',str(exc)[:300])
        if attempt<max_attempts:time.sleep(min(3*attempt,15))
    raise RuntimeError('Partial download after repeated attempts. Re-run this cell to resume from .part.')

def download_entire_archive_verified(url,final,part,out):
    final=Path(final);part=Path(part);out=Path(out)
    out.mkdir(parents=True,exist_ok=True)
    expected,etag=_remote_metadata(url)
    print('Publisher Content-Length:',expected if expected is not None else 'not supplied')
    if final.exists():
        print('Verifying existing candidate (full tar/gzip stream, including final 21.6-GB member)...')
        scan=validate_archive_structure(final)
        atomic_json(out/'ARCHIVE_PREFLIGHT.json',scan)
        if scan['complete'] and (expected is None or expected==final.stat().st_size):
            print('VALID COMPLETE ARCHIVE. Reusing existing file and 112+ checkpoints.')
            return final
        print('Invalid existing archive:',scan['stderr_tail'][-600:])
        if not AUTO_REPLACE_INVALID_PUBLIC_ARCHIVE:
            raise RuntimeError('Set AUTO_REPLACE_INVALID_PUBLIC_ARCHIVE=True after checking the stored archive.')
        atomic_json(out/'ARCHIVE_REPAIR_EVENT.json',{
            'old_file':str(final),'old_file_bytes':final.stat().st_size,
            'old_integrity':scan,'action':'remove_invalid_public_archive_and_redownload',
            'journal_retained':True})
        final.unlink()  # Only the invalid, publicly re-downloadable archive. Journal is NEVER deleted.
        print('Invalid archive removed to free Drive capacity. Old audit journal retained.')
    _stream_with_resume(url,part,expected,etag)
    size=part.stat().st_size
    if expected is not None and size!=expected:
        raise RuntimeError(f'Download incomplete: {size:,} vs {expected:,} source bytes')
    print('Checking entire gzip CRC and TAR integrity before promotion...')
    scan=validate_archive_structure(part)
    atomic_json(out/'ARCHIVE_PREFLIGHT.json',scan)
    if not scan['complete']:
        raise RuntimeError('Downloaded archive fails TAR/GZIP integrity even after transfer. '
            'Check ARCHIVE_PREFLIGHT.json. Do NOT run the full scan; re-download '
            'the .part from scratch or inspect the publisher source. '+scan['stderr_tail'][-600:])
    os.replace(part,final)
    print('COMPLETE VERIFIED ARCHIVE',final,'bytes:',final.stat().st_size)
    return final

ARCHIVE = download_entire_archive_verified(URL,ARCHIVE,PART,RESULTS)


## 3. Resume all-member HDF5 audit

In [ ]:
# Full dataset, all archive members, every numeric HDF5 value. Old journal retained.
# Download function above has already validated the complete TAR/GZIP stream.
# This call hashes the full archive, then reads each item; if the archive was repaired,
# cached OK items are individually SHA-256 revalidated before their numerical results are reused.
FULL_AUDIT = run_full_archive(ARCHIVE,RESULTS,scratch_dir=SCRATCH,strict=True,preflight=False)
assert FULL_AUDIT['status']=='COMPLETE_FULL_ARCHIVE_AUDIT'
assert FULL_AUDIT['members_ok']==FULL_AUDIT['archive_members_regular']
print(json.dumps(FULL_AUDIT,indent=2))


## 4. Verify the unchanged frozen RC3 model

In [ ]:
FROZEN_REF='frozen/rc3-sionna-qualified-2026-11-14'
FROZEN_COMMIT='182e77b485cceb770ee6570ba332205c73823c91'
ENGINE_DIR=Path('/content/SANAD-Network-RC3-FROZEN')
if not ENGINE_DIR.exists():
    subprocess.run(['git','clone','--depth','1','--branch',FROZEN_REF,
        'https://github.com/majdishokri/SANAD-Network-RC3.git',str(ENGINE_DIR)],check=True)
commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ENGINE_DIR,text=True).strip()
assert commit==FROZEN_COMMIT, f'RC3 frozen ref changed: {commit}'
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],cwd=ENGINE_DIR,check=True)
subprocess.run([sys.executable,'verify_freeze.py'],cwd=ENGINE_DIR,check=True)
subprocess.run([sys.executable,'qualification/verify_qualification.py'],cwd=ENGINE_DIR,check=True)
subprocess.run([sys.executable,'-c', 'from rc3 import FrozenRC3; e=FrozenRC3(); assert e.threshold==0.50; print("RC3 model loaded at threshold 0.50")'], cwd=ENGINE_DIR,check=True)
print('Frozen RC3 integrity and model-load check passed at',commit)

## 5. Save RC3 data-compatibility report

In [ ]:
COMPATIBILITY=rc3_contract_report(RESULTS,ENGINE_DIR)
final={
    'dataset':'Rice LensFD full 18-GB published archive',
    'source':'https://renew.rice.edu/dataset-lens.html',
    'dataset_full_coverage':FULL_AUDIT['status'],
    'source_archive_sha256':FULL_AUDIT['archive_sha256'],
    'all_archive_regular_files_processed':FULL_AUDIT['members_ok']==FULL_AUDIT['archive_members_regular'],
    'numeric_hdf5_values_scanned':FULL_AUDIT['hdf5_numeric_values_scanned'],
    'frozen_rc3_commit':commit,
    'frozen_rc3_threshold':0.50,
    'rc3_lensfd_adapter_status':COMPATIBILITY['adapter_status'],
    'rc3_fault_detection_rate_on_lensfd':None,
    'research_note':'Real measurements fully processed; no invented model inference features or accuracy claims.'
}
(RESULTS/'RUN_FINAL_STATUS.json').write_text(json.dumps(final,indent=2))
print(json.dumps(final,indent=2))
print('Complete results folder:',RESULTS)
assert final['all_archive_regular_files_processed']

### Files saved to your Google Drive

`SANAD_Network/RC3_LensFD_FULL/source_archive_PRIVATE/data_lensFD.tgz` (complete licensed original), `results_full/SOURCE_LOCK.json`, `results_full/members.jsonl` (every member, HDF5 statistics, SHA-256), `results_full/FULL_AUDIT_SUMMARY.json`, `results_full/RC3_COMPATIBILITY.json`, `results_full/RUN_FINAL_STATUS.json`.

**No subset, no random selection, no imputed timestamps/labels, no fake RC3 accuracy.** This notebook cannot be declared complete until every member has been accounted for.